In [1]:
import os
from dotenv import load_dotenv, find_dotenv
from openai import OpenAI

load_dotenv(find_dotenv(usecwd=True))

endpoint = os.getenv("AZURE_OPENAI_ENDPOINT", "").rstrip("/")
api_key = os.getenv("AZURE_OPENAI_API_KEY", "")
deployment = os.getenv("AZURE_OPENAI_DEPLOYMENT", "gpt-5.4-mini")

print("Endpoint set:", bool(endpoint))
print("API key set:", bool(api_key))
print("Deployment:", deployment)

client = OpenAI(base_url=f"{endpoint}/openai/v1/", api_key=api_key, timeout=60)

Endpoint set: True
API key set: True
Deployment: gpt-5.4-mini


In [2]:
response = client.chat.completions.create(
    model=deployment,
    messages=[{"role": "user", "content": "Say hello in ten short sentence."}],
    max_completion_tokens=50,
)
print(response.choices[0].message.content)

Hello.  
Hi there.  
Greetings.  
Good day.  
Hey!  
Nice to meet you.  
Welcome.  
Howdy.  
Salutations.  
Hello again.


In [3]:
review = "The paster was cold and the waiter ignored us, but the tiramisu was amazing."

In [11]:
system_prompt = f""" you are a helpful assistant that analyzes restaurant reviews and provides sentiment score from 1 to 5 where 1 is very negative and 5 is very positive. review :'{review}'"""

In [12]:
response = client.chat.completions.create(
    model=deployment,
    messages=[{"role": "user", "content": system_prompt}],
    max_completion_tokens=50,
)
print(response.choices[0].message.content)

Sentiment score: **2/5**

The review is mostly negative because of the cold pasta and inattentive service, but the amazing tiramisu adds a bit of positivity.


In [13]:
zero_shot_prompt = f""" you are a helpful assistant that analyzes restaurant reviews and provides sentiment score from 1 to 5 where 1 is very negative and 5 is very positive. review :'{review}', you also provide ouput of food ambience and service using the same scale."""

In [14]:
response = client.chat.completions.create(
    model=deployment,
    messages=[{"role": "user", "content": zero_shot_prompt}],
    max_completion_tokens=50,
)
print(response.choices[0].message.content)

Sentiment analysis for the review:

- Overall sentiment: **3/5**  
- Food: **3/5**  
- Ambience: **3/5**  
- Service: **1/5**

Reasoning:


In [15]:
review = "The paster was cold and the waiter ignored us, but the tiramisu was amazing."

zero_shot_system_prompt_v2 = f""" you are a helpful assistant that analyzes restaurant reviews and provides sentiment score from 1 to 5 where 1 is very negative and 5 is very positive. review :'{review}', you also provide ouput of food ambience and service using the same scale. please provide your output in json format with keys 'sentiment', 'food', 'ambience', 'service'. if ther is no information about any of the keys, please provide a value of 'N/A' for that catergory you will your output in json format with the following keys: 'overall', 'ambience', 'food', 'service'."""

response = client.chat.completions.create(
    model=deployment,
    messages=[{"role": "user", "content": zero_shot_system_prompt_v2}],
    max_completion_tokens=50,
)
print(response.choices[0].message.content)

{
  "overall": 3,
  "ambience": "N/A",
  "food": 3,
  "service": 1
}


In [20]:
review = "The paster was cold and the waiter ignored us, but the tiramisu was amazing."
chain_of_thoughts_prompt = f""" you are a helpful assistant that analyzes restaurant reviews and provides sentiment score from 1 to 5 where 1 is very negative and 5 is very positive. review :'{review}', you also provide ouput of food ambience and service using the same scale. please provide your output in json format with keys 'sentiment', 'food', 'ambience', 'service'. if ther is no information about any of the keys, please provide a value of 'N/A' for that catergory you will your output in json format with the following keys: 'overall', 'ambience', 'food', 'service'. please provide your reasoning before providing your final answer."""

response = client.chat.completions.create(
    model=deployment,
    messages=[{"role": "user", "content": chain_of_thoughts_prompt}],
    max_completion_tokens=500,
)
print(response.choices[0].message.content)

Reasoning: The review is mixed. The food had both a negative and positive aspect: the pasta was cold, but the tiramisu was amazing, so overall food is slightly positive. The waiter ignored the customers, which indicates poor service. There is no mention of the ambience, so that category is N/A. Overall sentiment is slightly negative to neutral because the service problem and cold pasta outweigh the dessert praise.

```json
{
  "overall": 2,
  "ambience": "N/A",
  "food": 3,
  "service": 1
}
```


In [19]:

Tree_of_Thoughts_prompt = f""" you are a helpful assistant that analyzes restaurant reviews and provides sentiment score from 1 to 5 where 1 is very negative and 5 is very positive. review :'{review}', you also provide ouput of food ambience and service using the same scale. please provide your output in json format with keys 'sentiment', 'food', 'ambience', 'service'. if ther is no information about any of the keys, please provide a value of 'N/A' for that catergory you will your output in json format with the following keys: 'overall', 'ambience', 'food', 'service'. please provide your reasoning before providing your final answer. please provide your reasoning in a step by step manner.all in json format with keys 'reasoning' and 'final_answer' where 'reasoning' is a list of steps and 'final_answer' is the final output in json format with keys 'overall', 'ambience', 'food', 'service'."""

response = client.chat.completions.create(
    model=deployment,
    messages=[{"role": "user", "content": Tree_of_Thoughts_prompt}],
    max_completion_tokens=500,
)
print(response.choices[0].message.content)

{
  "reasoning": [
    "The review says the pasta was cold, which is a negative comment about the food.",
    "The review says the waiter ignored us, which is a negative comment about service.",
    "The review says the tiramisu was amazing, which is a positive comment about food.",
    "There is no information about the ambience.",
    "Overall sentiment is mixed, but slightly negative because the bad service and cold pasta outweigh the positive dessert."
  ],
  "final_answer": {
    "overall": 2,
    "ambience": "N/A",
    "food": 3,
    "service": 2
  }
}


# LLM AS A JUDGE

In [24]:
review = "The paster was cold and the waiter ignored us, but the tiramisu was amazing."

judge_system_prompt = f""" you are a helpful assistant that analyzes restaurant reviews and providee a final evaluation with 'consistent'. review {'review_output'}"""

response = client.chat.completions.create(
    model=deployment,
    messages=[{"role": "user", "content": judge_system_prompt}],
    max_completion_tokens=500,
)
print(response.choices[0].message.content)

consistent


In [39]:
from pydantic import BaseModel, ValidationError, Field
from typing import List

class ReviewScores(BaseModel):
    overall: int
    ambience: str
    food: int
    service: int

class ReviewOutput(BaseModel):
    reasoning: List[str]
    final_answer: ReviewScores

In [40]:
response = client.chat.completions.create(
    model=deployment,
    messages=[{"role": "user", "content": Tree_of_Thoughts_prompt}],
    max_completion_tokens=500,
)
print(response.choices[0].message.content)
try:
    review_output = ReviewOutput.model_validate_json(response.choices[0].message.content)
    print("Parsed output:", review_output)
except ValidationError as e:
    print("Validation error:", e)
    print("Response content:", response.choices[0].message.content)

{
  "reasoning": [
    "The review says the pasta was cold, which is a negative comment about the food.",
    "The review says the tiramisu was amazing, which is a positive comment about the food.",
    "The review says the waiter ignored us, which is a negative comment about service.",
    "There is no clear mention of ambience in the review.",
    "Overall sentiment is mixed: one strong positive about dessert, but negative remarks about the pasta and service. The positives and negatives balance to a mildly negative to neutral impression."
  ],
  "final_answer": {
    "overall": 3,
    "ambience": "N/A",
    "food": 3,
    "service": 1
  }
}
Parsed output: reasoning=['The review says the pasta was cold, which is a negative comment about the food.', 'The review says the tiramisu was amazing, which is a positive comment about the food.', 'The review says the waiter ignored us, which is a negative comment about service.', 'There is no clear mention of ambience in the review.', 'Overall s

In [38]:
print(review_output.final_answer.overall)
print(review_output.final_answer.ambience)
print(review_output.final_answer.food)
print(review_output.final_answer.service)

3
N/A
3
1
